# 03 - PHASE 0 GATE: decomposition of the efficiency gap on REAL logits

AGENTS.md Sec 5. The evidence that motivated this project came from synthetic
inject-then-recover, which is circular. Before anything else the decomposition must
replicate on REAL logits.

Measure how much of the efficiency gap is closed, SEPARATELY, by:
1. a single global **temperature** (1 free parameter),
2. a per-sample offset indexed by **free energy** `E(x) = -logsumexp(logits)` (n_bins params),
3. a per-**class** offset fit on abundant data (K params).

**PRE-REGISTERED PASS CRITERION (Sec 5):** (3) must close a gap SUBSTANTIALLY larger than
(1) and (2), with **non-overlapping CIs**. If it does not, the hypothesis that the structure
lives at the class level does not hold on real data -> **STOP, do not enter Phase 1.**

All three use the SAME conformal offset estimator on abundant data, differing only in what
INDEXES the correction (global / per-sample-energy / per-class), so the comparison is
apples-to-apples. The energy component is swept over bin counts because a component with
more free parameters must not win on parameter count alone.

**CIFAR-100 caveat:** this is the pipeline-DEBUG dataset (100 classes, 100 test images/class,
alpha=0.01 infeasible - see reports/phase0_checkpoint_gate.md). A CIFAR-100 result does NOT
decide the Phase-0 gate; Pl@ntNet does. Here we verify the code and read the direction.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'cifar100'
BACKBONE   = 'resnet50_self'

ALPHAS     = (0.05, 0.1)        # alpha=0.01 is infeasible at 100 img/class (pre-registered)
N_SPLITS   = 100               # >=100 random cal/eval splits (Sec 8.4)
BIN_GRID   = (2, 5, 10, 20, 50, 100)
SEED = 42
EMB_DIR = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_DIR =', EMB_DIR, '| alphas', ALPHAS)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load TEST-set logits (the conformal cal/eval pool)


In [ ]:
import numpy as np, os
d = np.load(os.path.join(EMB_DIR,'test.npz'))
logits, labels = d['logits'], d['labels']
n, K = logits.shape
print(f'logits={logits.shape} classes={K} samples/class~{n//K}')
print('accuracy:', round(float((logits.argmax(1)==labels).mean()),4))


## 4. Run the decomposition over >=100 random cal/eval splits (Sec 8.4)


In [ ]:
from pcc.eval import decomposition as dc
from pcc.eval.stats import mean_ci
import numpy as np

results = {}
for alpha in ALPHAS:
    acc = {'temperature': [], 'class_offset': [],
           **{f'energy_b{b}': [] for b in BIN_GRID}}
    S = 1 - dc.temperature_softmax(logits, 1.0)
    rng = np.random.default_rng(SEED)
    for s in range(N_SPLITS):
        idx = rng.permutation(n); cal, ev = idx[:n//2], idx[n//2:]
        acc['temperature'].append(
            dc.gap_from_global_temperature(logits, labels, alpha, cal, ev)['gap_closed'])
        acc['class_offset'].append(
            dc.gap_from_per_class_offset(S, labels, K, alpha, cal, ev)['gap_closed'])
        sw = dc.phase0_energy_bin_sweep(logits, S, labels, alpha, cal, ev, bin_grid=BIN_GRID)
        for b in BIN_GRID:
            acc[f'energy_b{b}'].append(sw[b]['gap_closed'])
    results[alpha] = {k: mean_ci(v) for k, v in acc.items()}
    print(f'--- alpha={alpha} ---')
    for k, v in results[alpha].items():
        print(f"  {k:16s} gap={v['mean']:+7.3f}  95% CI [{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]")


## 5. Gate verdict - non-overlapping CIs (Sec 5)


In [ ]:
verdicts = {}
for alpha in ALPHAS:
    r = results[alpha]
    cls = r['class_offset']
    rivals = {k: v for k, v in r.items() if k != 'class_offset'}
    best_name = max(rivals, key=lambda k: rivals[k]['mean'])
    best = rivals[best_name]
    # substantially larger AND CIs do not overlap
    non_overlap = cls['ci_low'] > best['ci_high']
    verdicts[alpha] = {'class_gap': cls['mean'], 'best_rival': best_name,
                       'rival_gap': best['mean'], 'non_overlapping_CI': bool(non_overlap),
                       'pass': bool(non_overlap and cls['mean'] > best['mean'])}
    print(f"alpha={alpha}: class={cls['mean']:+.3f} [{cls['ci_low']:+.3f},{cls['ci_high']:+.3f}] "
          f"vs best rival {best_name}={best['mean']:+.3f} [{best['ci_low']:+.3f},{best['ci_high']:+.3f}] "
          f"-> {'PASS' if verdicts[alpha]['pass'] else 'FAIL'}")

overall = 'PASS' if all(v['pass'] for v in verdicts.values()) else 'FAIL'
print('\nCIFAR-100 (DEBUG) direction:', overall)
print('NOTE: this does NOT decide the Phase-0 gate - Pl@ntNet does (see cell 0).')


## 6. Write report


In [ ]:
import time, json
from pcc.utils.io import write_report
clean = {str(a): {k: v for k, v in r.items()} for a, r in results.items()}
for a in clean:
    for k in clean[a]:
        clean[a][k] = {kk: (float(vv) if hasattr(vv,'__float__') else vv)
                       for kk, vv in clean[a][k].items()}
report = write_report('pcc/reports', f'03_phase0_decomposition_{DATASET}',
    hypothesis='a per-CLASS offset closes a substantially larger efficiency gap than a global '
               'temperature or a per-sample energy-indexed offset, on REAL logits',
    pass_criteria='class_offset gap > best rival AND non-overlapping 95% CIs, at every alpha; '
                  'energy component swept over bin counts so the win is not a parameter-count '
                  'artefact. CIFAR-100 is DEBUG ONLY and does not decide the gate.',
    config=dict(dataset=DATASET, backbone=BACKBONE, alphas=list(ALPHAS),
                n_splits=N_SPLITS, bin_grid=list(BIN_GRID)),
    seed=SEED, results={'by_alpha': clean, 'verdicts': {str(k): v for k, v in verdicts.items()},
                        'debug_only': True},
    conclusion=f'{overall} (CIFAR-100 debug direction; not the gate verdict)',
    started_at=time.time())
print('report:', report)
